<a href="https://colab.research.google.com/github/MusicalManiac/SatelliteDataAI-UOA/blob/main/Lab3_Answers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Code to question 2
import warnings
warnings.filterwarnings('ignore')

import urllib.request
from PIL import Image
import matplotlib.pyplot as plt

AOI = ee.Geometry.Point(174.7633, -36.8485)
START_DATE = '2023-01-01'
END_DATE = '2023-03-31'

def apply_cld_shdw_mask(img):
    cloudmask = img.select('cloudmask')
    masked_img = img.updateMask(cloudmask.Not())
    cloudmask_band = img.select('cloudmask')
    probability_band = img.select('probability')
    masked_img = masked_img.addBands([cloudmask_band, probability_band], overwrite=True)
    return masked_img

# BEFORE
CLD_PRB_THRESH = 50
NIR_DRK_THRESH = 0.15
CLD_PRJ_DIST = 1
BUFFER = 50

s2_before = get_s2_sr_cld_col(AOI, START_DATE, END_DATE)
s2_before_masked = (s2_before.map(add_cld_shdw_mask)
                              .map(apply_cld_shdw_mask)
                              .median())

# AFTER
CLD_PRB_THRESH = 40
NIR_DRK_THRESH = 0.20
CLD_PRJ_DIST = 1.5
BUFFER = 100

s2_after = get_s2_sr_cld_col(AOI, START_DATE, END_DATE)
s2_after_masked = (s2_after.map(add_cld_shdw_mask)
                            .map(apply_cld_shdw_mask)
                            .median())



plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# Control variables
AOI = ee.Geometry.Point(174.7633, -36.8485)
START_DATE = '2023-01-01'
END_DATE = '2023-03-31'

# BEFORE
CLD_PRB_THRESH = 50
NIR_DRK_THRESH = 0.15
CLD_PRJ_DIST = 1
BUFFER = 50

s2_before_col = get_s2_sr_cld_col(AOI, START_DATE, END_DATE).map(add_cld_shdw_mask)
frac_before = s2_before_col.select('cloudmask').mean()   # fraction of dates flagged cloudy/shadow, 0-1
rgb_before = s2_before_col.map(apply_cld_shdw_mask).median()

# AFTER
CLD_PRB_THRESH = 35
NIR_DRK_THRESH = 0.20
CLD_PRJ_DIST = 1.5
BUFFER = 100

s2_after_col = get_s2_sr_cld_col(AOI, START_DATE, END_DATE).map(add_cld_shdw_mask)
frac_after = s2_after_col.select('cloudmask').mean()
rgb_after = s2_after_col.map(apply_cld_shdw_mask).median()

def apply_cld_shdw_mask(img):
    cloudmask = img.select('cloudmask')
    masked_img = img.updateMask(cloudmask.Not())
    masked_img = masked_img.addBands([cloudmask, img.select('probability')], overwrite=True)
    return masked_img

# Thumbnail helper: RGB base + fraction-masked overlay
vis_params = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 2500, 'gamma': 1.1}

def get_frac_overlay_thumb(rgb_img, frac_img, region, dimensions=800):
    rgb = rgb_img.visualize(**vis_params)
    frac_vis = frac_img.visualize(min=0, max=1, palette=['00000000', 'ffff00', 'ff0000'])
    combined = rgb.blend(frac_vis.updateMask(frac_img.gt(0.05)))  # only show where >5% of dates flagged
    url = combined.getThumbURL({'region': region, 'dimensions': dimensions, 'format': 'png'})
    return Image.open(urllib.request.urlopen(url))

forest_region = ee.Geometry.Rectangle([174.45, -36.95, 174.60, -36.85])

img_before = get_frac_overlay_thumb(rgb_before, frac_before, forest_region)
img_after = get_frac_overlay_thumb(rgb_after, frac_after, forest_region)

# Build figure
fig, axes = plt.subplots(1, 2, figsize=(12, 6), dpi=300)
axes[0].imshow(img_before)
axes[0].set_title('Before (default parameters)', fontsize=12)
axes[0].axis('off')

axes[1].imshow(img_after)
axes[1].set_title('After (tuned parameters)', fontsize=12)
axes[1].axis('off')

caption = (
    "Figure 1. Fraction of dates flagged as cloud/shadow (yellow=low, red=high) "
    "over the Waitakere Ranges: default parameters (left) vs. tuned parameters "
    "(right; CLD_PRB_THRESH 50\u219240, NIR_DRK_THRESH 0.15\u21920.20, CLD_PRJ_DIST "
    "1\u21921.5, BUFFER 50\u2192100). Tuning reduced cloud contamination elsewhere in "
    "the scene but increased false-positive shadow flagging over this "
    "self-shadowed forest canopy."
)
fig.text(0.5, -0.03, caption, ha='center', va='top', wrap=True, fontsize=9)

plt.tight_layout()
plt.savefig('q2_before_after.png', dpi=300, bbox_inches='tight')
plt.savefig('q2_before_after.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Code to Question 6
import warnings
warnings.filterwarnings('ignore')

import ee
import geemap
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import urllib.request
from PIL import Image

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

#adding functions earlier used so if during restart , can directly run this
def get_s2_sr_cld_col(aoi, start_date, end_date):
    s2_sr_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(aoi).filterDate(start_date, end_date)
        .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', CLOUD_FILTER)))
    s2_cloudless_col = (ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY')
        .filterBounds(aoi).filterDate(start_date, end_date))
    return ee.ImageCollection(ee.Join.saveFirst('s2cloudless').apply(**{
        'primary': s2_sr_col, 'secondary': s2_cloudless_col,
        'condition': ee.Filter.equals(**{'leftField': 'system:index', 'rightField': 'system:index'})
    }))

def add_cloud_bands(img):
    cld_prb = ee.Image(img.get('s2cloudless')).select('probability')
    is_cloud = cld_prb.gt(CLD_PRB_THRESH).rename('clouds')
    return img.addBands(ee.Image([cld_prb, is_cloud]))

def add_shadow_bands(img):
    not_water = img.select('SCL').neq(6)
    SR_BAND_SCALE = 1e4
    dark_pixels = img.select('B8').lt(NIR_DRK_THRESH*SR_BAND_SCALE).multiply(not_water).rename('dark_pixels')
    shadow_azimuth = ee.Number(90).subtract(ee.Number(img.get('MEAN_SOLAR_AZIMUTH_ANGLE')))
    cld_proj = (img.select('clouds').directionalDistanceTransform(shadow_azimuth, CLD_PRJ_DIST*10)
        .reproject(**{'crs': img.select(0).projection(), 'scale': 100})
        .select('distance').mask().rename('cloud_transform'))
    shadows = cld_proj.multiply(dark_pixels).rename('shadows')
    return img.addBands(ee.Image([dark_pixels, cld_proj, shadows]))

def add_cld_shdw_mask(img):
    img_cloud = add_cloud_bands(img)
    img_cloud_shadow = add_shadow_bands(img_cloud)
    is_cld_shdw = img_cloud_shadow.select('clouds').add(img_cloud_shadow.select('shadows')).gt(0)
    is_cld_shdw = (is_cld_shdw.focalMin(2).focalMax(BUFFER*2/20)
        .reproject(**{'crs': img.select([0]).projection(), 'scale': 20})
        .rename('cloudmask'))
    return img_cloud_shadow.addBands(is_cld_shdw)

def apply_cld_shdw_mask(img):
    cloudmask = img.select('cloudmask')
    masked_img = img.updateMask(cloudmask.Not())
    masked_img = masked_img.addBands([cloudmask, img.select('probability')], overwrite=True)
    return masked_img

def sample_zones(img, aoi, edge_buffer=20):
    cloudmask = img.select('cloudmask')
    cloud_interior = cloudmask.selfMask()
    clear_interior = cloudmask.Not().selfMask()
    cloud_edge = cloudmask.focal_max(edge_buffer).And(cloudmask.Not())
    sample_bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B9', 'B8', 'B11', 'B12']
    if 'probability' in img.bandNames().getInfo():
        sample_bands.append('probability')
    img_for_sampling = img.select(sample_bands)
    cloud_samples = img_for_sampling.updateMask(cloud_interior).sample(
        region=aoi.buffer(2000), scale=10, numPixels=5000, seed=0, geometries=True)
    clear_samples = img_for_sampling.updateMask(clear_interior).sample(
        region=aoi.buffer(2000), scale=10, numPixels=5000, seed=1, geometries=True)
    edge_samples = img_for_sampling.updateMask(cloud_edge).sample(
        region=aoi.buffer(2000), scale=10, numPixels=5000, seed=2, geometries=True)
    return cloud_samples, clear_samples, edge_samples

def fc_to_df(fc):
    features = fc.getInfo()['features']
    rows = []
    for f in features:
        props = f['properties']
        coords = f['geometry']['coordinates']
        props['longitude'] = coords[0]
        props['latitude'] = coords[1]
        rows.append(props)
    return pd.DataFrame(rows)

vis_params = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 2500, 'gamma': 1.1}
START_DATE = '2023-01-01'
END_DATE = '2023-03-31'

# Multi-Region Sampling
aois = {
    'urban': ee.Geometry.Point(174.7633, -36.8485),
    'forest': ee.Geometry.Point(174.5200, -36.9200),
    'farmland': ee.Geometry.Point(174.9000, -37.0500),
    'coastal': ee.Geometry.Point(174.8300, -36.7900),
    'suburban': ee.Geometry.Point(174.7000, -36.9000),
}

CLOUD_FILTER = 30
CLD_PRB_THRESH = 30
NIR_DRK_THRESH = 0.15
CLD_PRJ_DIST = 1
BUFFER = 5

all_cloud_df, all_clear_df, all_edge_df = [], [], []

for name, aoi in aois.items():
    s2_col = get_s2_sr_cld_col(aoi, START_DATE, END_DATE)
    s2_mosaic = (s2_col.map(add_cld_shdw_mask).map(apply_cld_shdw_mask).mosaic())
    cloud_s, clear_s, edge_s = sample_zones(s2_mosaic, aoi)
    cdf, cldf, edf = fc_to_df(cloud_s), fc_to_df(clear_s), fc_to_df(edge_s)
    cdf['landcover'] = name
    cldf['landcover'] = name
    edf['landcover'] = name
    all_cloud_df.append(cdf)
    all_clear_df.append(cldf)
    all_edge_df.append(edf)
    print(f"{name}: cloud={len(cdf)}, clear={len(cldf)}, edge={len(edf)}")

cloud_df = pd.concat(all_cloud_df, ignore_index=True)
clear_df = pd.concat(all_clear_df, ignore_index=True)
edge_df = pd.concat(all_edge_df, ignore_index=True)

cloud_df['class'] = 'Cloud'
clear_df['class'] = 'Clear'
combined_df = pd.concat([cloud_df, clear_df])
min_count = combined_df['class'].value_counts().min()
balanced_df = combined_df.groupby('class').sample(n=min_count, random_state=42)

# 1st Figure
bands = ['B8', 'B9', 'B11']
fig1 = sns.pairplot(balanced_df, vars=bands, hue='class',
                     plot_kws={'alpha': 0.4, 's': 15}, height=3)
fig1.fig.suptitle('Multi-region training sample: Cloud vs Clear across land-cover types', y=1.02)
plt.savefig('q6_fig1_multiregion_sampling.png', dpi=300, bbox_inches='tight')
plt.savefig('q6_fig1_multiregion_sampling.pdf', bbox_inches='tight')
plt.show()

caption_fig1 = (
    "Figure 2. Cloud vs Clear separation in B8/B9/B11 using training samples "
    "drawn from five land-cover types (urban, forest, farmland, coastal, "
    "suburban) rather than downtown Auckland alone."
)
print(caption_fig1)

# Whole-image classification via cluster centres
features = ['B8', 'B9', 'B11']
train_df = balanced_df[balanced_df['class'].isin(['Cloud', 'Clear'])]
X_train = train_df[features].values
y_train = (train_df['class'] == 'Cloud').astype(int).values

kmeans = KMeans(n_clusters=2, random_state=42).fit(X_train)

labels, counts = np.unique(kmeans.labels_[y_train == 0], return_counts=True)
clear_cluster = labels[np.argmax(counts)]
labels, counts = np.unique(kmeans.labels_[y_train == 1], return_counts=True)
cloud_cluster = labels[np.argmax(counts)]

centres = kmeans.cluster_centers_
clear_centre = train_df[train_df['class'] == 'Clear'][features].mean().values
cloud_centre = train_df[train_df['class'] == 'Cloud'][features].mean().values
print("Clear centre:", clear_centre, "| Cloud centre:", cloud_centre)

def classify_whole_image(img):
    b8, b9, b11 = img.select('B8'), img.select('B9'), img.select('B11')
    dist_clear = (b8.subtract(clear_centre[0])).pow(2) \
                 .add((b9.subtract(clear_centre[1])).pow(2)) \
                 .add((b11.subtract(clear_centre[2])).pow(2)).sqrt()
    dist_cloud = (b8.subtract(cloud_centre[0])).pow(2) \
                 .add((b9.subtract(cloud_centre[1])).pow(2)) \
                 .add((b11.subtract(cloud_centre[2])).pow(2)).sqrt()
    return img.addBands(dist_cloud.lt(dist_clear).rename('kmeans_cloud'))

def classify_near_clouds_only(img, buffer_px=20):
    cloudmask = img.select('cloudmask')
    search_zone = cloudmask.focal_max(buffer_px).And(cloudmask.Not())
    classified = classify_whole_image(img)
    restricted = classified.select('kmeans_cloud').updateMask(search_zone)
    return img.addBands(restricted.rename('kmeans_cloud_restricted'))

test_aoi = aois['urban']
s2_col_test = get_s2_sr_cld_col(test_aoi, START_DATE, END_DATE)
s2_mosaic_test = (s2_col_test.map(add_cld_shdw_mask).map(apply_cld_shdw_mask).mosaic())
classified_img = classify_near_clouds_only(s2_mosaic_test)

def get_kmeans_overlay_thumb(img, band, region, dimensions=800):
    rgb = img.visualize(**vis_params)
    kmeans_vis = img.select(band).selfMask().visualize(palette=['00ffff'])
    combined = rgb.blend(kmeans_vis)
    url = combined.getThumbURL({'region': region, 'dimensions': dimensions, 'format': 'png'})
    return Image.open(urllib.request.urlopen(url))

whole_scene_region = test_aoi.buffer(15000).bounds()
img_fig2 = get_kmeans_overlay_thumb(classified_img, 'kmeans_cloud_restricted', whole_scene_region)

# 2nd Figure
plt.figure(figsize=(10, 8), dpi=300)
plt.imshow(img_fig2)
plt.axis('off')
plt.title('K-means classification (nearest cluster centre) applied across whole scene', fontsize=12)
plt.tight_layout()
plt.savefig('q6_fig2_whole_scene_classification.png', dpi=300, bbox_inches='tight')
plt.savefig('q6_fig2_whole_scene_classification.pdf', bbox_inches='tight')
plt.show()

caption_fig2 = (
    f"Figure 3. Class-mean spectral centroids (Cloud: {cloud_centre.round(0)}; "
    f"Clear: {clear_centre.round(0)}) were passed into Earth Engine as "
    f"constants and used to classify pixels by nearest centroid within a zone "
    f"around existing cloud detections. The classifier also flags bright "
    f"urban surfaces (roads, rooftops) as cloud-like, reflecting the same "
    f"NIR/SWIR brightness overlap between built material and cloud discussed "
    f"in Q3 — a limitation of relying on only three spectral bands."
)
print(caption_fig2)